# DeployTeach Pyannote GPU Evaluation

Use this notebook with a zip produced by `tools/package_pyannote_colab_eval.py`. The zip should contain `generated_dataset/audio`, `generated_dataset/vad`, `generated_dataset/truth`, and `tools/`.

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_name = next(iter(uploaded))
print(zip_name)

In [ ]:
!rm -rf /content/deployteach_eval
!mkdir -p /content/deployteach_eval
!unzip -q "{zip_name}" -d /content/deployteach_eval
%cd /content/deployteach_eval

In [ ]:
!find . -maxdepth 3 -type d | sort
!python3 - <<'PY'
from pathlib import Path
for name, pattern in [('audio', 'generated_dataset/audio/*.wav'), ('vad', 'generated_dataset/vad/*.json'), ('truth', 'generated_dataset/truth/*.json')]:
    print(name, len(list(Path('.').glob(pattern))))
PY

In [ ]:
!pip install -q pyannote.audio==3.3.2

In [ ]:
import os
import torch
from google.colab import userdata

token = userdata.get('HF_TOKEN')
if not token:
    raise RuntimeError('Add HF_TOKEN in Colab secrets before running this cell.')
os.environ['HF_TOKEN'] = token

print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))

In [ ]:
!python3 tools/run_pyannote_baseline.py \
  --audio generated_dataset/audio \
  --truth generated_dataset/truth \
  --vad generated_dataset/vad \
  --out generated_dataset/pyannote_android_pipeline \
  --device cuda \
  --score

In [ ]:
!cat generated_dataset/pyannote_android_pipeline/metrics.json
!zip -qr pyannote_android_pipeline_results.zip generated_dataset/pyannote_android_pipeline
files.download('pyannote_android_pipeline_results.zip')